# PAUL Open Model — First Model Validation (Gemma 4 E4B IT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/foundrypaul-cloud/paul-open/blob/main/notebooks/02_first_model_validation_e4b.ipynb)

This notebook executes the **First Model Validation Experiment** on Google Colab (Tesla T4, 14.56 GiB usable VRAM).

### Experiment Scope & Constraints
- **Target Model**: `google/gemma-4-E4B-it` (~4.5B dense parameters, edge-optimized multimodal).
- **Architecture**: `AutoModelForMultimodalLM` (official Gemma 4 Transformers API).
- **Quantization**: 4-bit NF4 QLoRA-compatible configuration via `BitsAndBytesConfig` (`dtype=torch.float16`).
- **Hardware**: Tesla T4 (14.56 GiB usable VRAM).
- **Safety Rules**:
  - No full-precision / BF16 loading.
  - No training or dataset downloads.
  - No Hugging Face model repository creation.
  - Complete GPU memory cleanup and peak VRAM audit after inference.

## Step 1: Repository Setup & Explicit Package Installation
Clone the repository (if in Colab) and install the verified, pinned dependency stack.

In [ ]:
import os
import sys

# In Google Colab, clone the repository to access local configs and src package
if "google.colab" in sys.modules or os.environ.get("COLAB_GPU") is not None:
    if not os.path.exists("paul-open"):
        !git clone -q https://github.com/foundrypaul-cloud/paul-open.git
        %cd paul-open
    elif os.path.basename(os.getcwd()) != "paul-open":
        %cd paul-open

# Install explicit, pinned versions of the Gemma 4 ML stack
!pip install -q --no-cache-dir \
    "torch>=2.11.0" \
    "transformers>=5.13.1" \
    "peft>=0.19.0" \
    "trl>=1.9.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.2.0" \
    "huggingface_hub>=0.28.0" \
    "tensorboard>=2.18.0" \
    "pyyaml>=6.0" \
    "rich>=13.0.0" \
    "sentencepiece>=0.2.0" \
    "tokenizers>=0.21.0"

## Step 2: Hugging Face Authentication & Model Access Verification
Retrieve the `HF_TOKEN` from Colab Secrets and verify access permissions for `google/gemma-4-E4B-it` before loading.

In [ ]:
import os
from huggingface_hub import HfApi, login

# Retrieve HF_TOKEN from Google Colab Secrets or Environment
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found! Please add your Hugging Face token in Colab Secrets (key icon on left sidebar) "
        "with key name 'HF_TOKEN' and grant access to this notebook."
    )

login(token=hf_token, add_to_git_credential=False)
print("✓ Hugging Face authenticated successfully.")

MODEL_ID = "google/gemma-4-E4B-it"
api = HfApi()
try:
    model_info = api.model_info(MODEL_ID, token=hf_token)
    print(f"✓ Verified access to {MODEL_ID}")
    print(f"  - Pipeline tag : {model_info.pipeline_tag}")
    print(f"  - Tags         : {model_info.tags[:8]}...")
except Exception as e:
    raise RuntimeError(
        f"Failed to access {MODEL_ID}. Please ensure you have accepted the Gemma license terms at "
        f"https://huggingface.co/{MODEL_ID}"
    ) from e

## Step 3: Hardware Baseline & 4-bit QLoRA Model Loading
Record initial GPU memory, configure 4-bit NF4 quantization, and load model and processor via the official Gemma 4 Transformers API using `AutoModelForMultimodalLM`.

In [ ]:
import gc
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

def print_gpu_memory(label: str):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        max_alloc = torch.cuda.max_memory_allocated() / (1024 ** 3)
        print(f"[{label}] Allocated: {allocated:.2f} GiB | Reserved: {reserved:.2f} GiB | Peak: {max_alloc:.2f} GiB")
    else:
        print(f"[{label}] CUDA not available (running on CPU)")

print_gpu_memory("Baseline Before Loading")

# Configure 4-bit NF4 QLoRA quantization
# On Tesla T4 (Turing architecture, CC 7.5), compute dtype float16 is used
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Using compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

print(f"Loading {MODEL_ID} weights using AutoModelForMultimodalLM in 4-bit NF4...")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=hf_token,
)

print_gpu_memory("Post Model Loading")

## Step 4: Chat Template & Inference Verification
Run one test inference using the official Gemma 4 chat template formatting with `enable_thinking=False` for baseline verification.

In [ ]:
# Define test educational query
messages = [
    {
        "role": "user",
        "content": "Explain what photosynthesis is in two clear sentences for a middle school science student."
    }
]

# Format input using Gemma 4 native chat template (enable_thinking=False for baseline)
formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("--- Formatted Input Prompt ---")
print(formatted_prompt)
print("------------------------------")

inputs = processor(text=formatted_prompt, return_tensors="pt").to(model.device)

print("Running inference generation...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

# Slice generated tokens past the input prompt
input_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_len:]
response = processor.decode(generated_tokens, skip_special_tokens=True)

print("\n--- Model Response ---")
print(response.strip())
print("----------------------")
print_gpu_memory("Post Inference")

## Step 5: GPU Cleanup & Final Validation Summary
Release GPU tensors, clear CUDA cache, and report peak VRAM footprint.

In [ ]:
# Record peak memory
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

# Cleanup model and tensors
del model
del processor
del inputs
del outputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("=" * 70)
print(" FIRST MODEL VALIDATION EXPERIMENT SUMMARY")
print("=" * 70)
print(f" Target Model       : {MODEL_ID}")
print(f" API Architecture   : AutoModelForMultimodalLM")
print(f" Quantization Mode  : 4-bit NF4 (Double Quantization)")
print(f" Peak VRAM Observed : {peak_vram_gb:.2f} GiB (Fits comfortably in 14.56 GiB T4)")
print(" Chat Template Test : PASSED (enable_thinking=False)")
print(" Inference Gen Test : PASSED")
print(" Cleanup Status     : GPU memory successfully released")
print("=" * 70)